<a href="https://colab.research.google.com/github/EricSnunes/AnaliseDeDados/blob/main/Captura_De_livros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importando as bibliotecas necessárias
import requests                # Para fazer requisições HTTP e obter o HTML da página
from bs4 import BeautifulSoup  # Para "parsear" o HTML e navegar pelas tags
import pandas as pd            # Para organizar os dados em tabelas (DataFrame)
import re                      # Para usar expressões regulares e limpar os preços

# 1. Fazendo a requisição ao site
url = "https://books.toscrape.com/"
response = requests.get(url)

# 2. Verificando se a requisição foi bem-sucedida
if response.status_code == 200:
    # 3. Transformando o HTML em objeto navegável com BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")

    # 4. Selecionando todos os blocos de livros (cada livro está dentro de <article class="product_pod">)
    books = soup.find_all("article", class_="product_pod")

    # 5. Criando uma lista para armazenar os dados
    data = []
    for book in books:
        # Pegando o título do livro (atributo "title" dentro da tag <a>)
        title = book.h3.a["title"]

        # Pegando o preço (texto dentro da tag <p class="price_color">)
        price = book.find("p", class_="price_color").get_text()

        # Pegando a disponibilidade (texto dentro da tag <p class="instock availability">)
        availability = book.find("p", class_="instock availability").get_text().strip()

        # Adicionando os dados coletados à lista
        data.append({"Título": title, "Preço": price, "Disponibilidade": availability})

    # 6. Transformando a lista em um DataFrame do Pandas
    df = pd.DataFrame(data)
    print("Tabela de livros coletados:\n")
    print(df.head(50))

    # 7. Limpando os preços para converter em número
    # Usamos regex para extrair apenas o número (ex.: 51.77)
    df["Preço_num"] = df["Preço"].apply(lambda x: float(re.findall(r"\d+\.\d+", x)[0]))

    # 8. Fazendo análises simples
    print("\nPreço médio dos livros:", round(df["Preço_num"].mean(), 2))
    print("Livro mais caro:", df.loc[df["Preço_num"].idxmax(), "Título"])
    print("Livro mais barato:", df.loc[df["Preço_num"].idxmin(), "Título"])

else:
    print("Erro na requisição:", response.status_code)


Tabela de livros coletados:

                                               Título    Preço Disponibilidade
0                                A Light in the Attic  Â£51.77        In stock
1                                  Tipping the Velvet  Â£53.74        In stock
2                                          Soumission  Â£50.10        In stock
3                                       Sharp Objects  Â£47.82        In stock
4               Sapiens: A Brief History of Humankind  Â£54.23        In stock
5                                     The Requiem Red  Â£22.65        In stock
6   The Dirty Little Secrets of Getting Your Dream...  Â£33.34        In stock
7   The Coming Woman: A Novel Based on the Life of...  Â£17.93        In stock
8   The Boys in the Boat: Nine Americans and Their...  Â£22.60        In stock
9                                     The Black Maria  Â£52.15        In stock
10     Starving Hearts (Triangular Trade Trilogy, #1)  Â£13.99        In stock
11                     